# Chapter 3 · Prompt Engineering for Agents

This notebook walks through the chapter section by section.
Every cell maps to a numbered section so you can read the chapter
and run the code side by side.

**Sections**

- 3.1  Prompt Engineering for Agents
- 3.2  Principles for Multi-Agent Prompting
- 3.3  Building a Prompting Framework (incl. Fact-Checker)
- 3.4  Agent Skills: Reusable Procedural Knowledge
- 3.5  Prompt Optimisation with DSPy
- 3.6  Structured Outputs
- 3.7  The Full Pattern: Skills + Structured Outputs

> **Context Engineering** has moved to Chapter 5 — it belongs alongside
> the memory systems that make dynamic context assembly possible.


## 3.0 · Setup

In [ ]:
import dspy
from dspy.teleprompt import MIPROv2
import os
import re
import json
from pathlib import Path
from typing import Any
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError

load_dotenv()


## 3.1 · Prompt Engineering for Agents

The system prompt is assembled from four labelled sections:
`[ROLE]`, `[SKILL]`, `[CONSTRAINTS]`, `[FORMAT]`.
Each section serves a distinct function — and labelling them is not
cosmetic. Language models respond to structure, and clearly named
sections help the model apply the right rules at the right time.

In [ ]:
SKILL_PATH = Path("SKILL.md")

def load_skill(path: Path = SKILL_PATH) -> str:
    """
    Load the Nutrition Assessment Protocol from SKILL.md.
    Returns an empty string if the file is not found —
    the agent continues to function, degraded but not crashed.
    """
    try:
        text = path.read_text()
        if text.startswith("---"):
            parts = text.split("---", 2)
            if len(parts) == 3:
                return parts[2].strip()
        return text
    except FileNotFoundError:
        return ""

NUTRITION_ASSESSMENT_SKILL = load_skill()

def build_system_prompt(skill: str = NUTRITION_ASSESSMENT_SKILL) -> str:
    return f"""[ROLE]
You are an AI Diet Coach with formal training in nutrition science.
Your mission: help users build sustainable, evidence-based eating habits.

[SKILL — Nutrition Assessment Protocol]
{skill}

[CONSTRAINTS]
- Never diagnose medical conditions. Refer users to a registered dietitian
  for therapeutic diets (e.g. renal diet, eating disorder recovery).
- Do not fabricate specific calorie counts for branded or restaurant foods.
- Do not recommend supplements above safe upper intake levels.
- If a user reports symptoms (dizziness, nausea, rapid weight loss), advise
  them to seek medical attention immediately.

[FORMAT]
When conducting a nutrition assessment, follow the four steps in the Skill
exactly (Baseline -> Deficits -> Priorities -> Single Goal).
For casual questions, reply conversationally in 2-4 sentences.
Always end assessments with a clearly labelled 'Your Goal This Week:' section.
"""

# Inspect the loaded Skill
print(f"Skill loaded: {len(NUTRITION_ASSESSMENT_SKILL.splitlines())} lines")
print(build_system_prompt()[:300] + "...")


## 3.2 · Principles for Multi-Agent Prompting

In a multi-agent system the orchestrator's output contract is as important
as its routing logic. Requiring a JSON response with fixed keys (`agent`, `query`)
means the routing decision can be parsed and acted on programmatically —
not interpreted from prose.

In [ ]:
ORCHESTRATOR_PROMPT = """You are the Diet Coach Orchestrator.
Your job is to route user requests to the correct specialist sub-agent:

  • NutritionAnalyst  — detailed macro/micro-nutrient breakdowns
  • MealPlanner       — weekly meal plans and recipe suggestions
  • BehaviourCoach    — habit formation, motivation, accountability

Decide which agent to call, pass only the relevant context, and
synthesise their outputs into a single coherent reply for the user.
Return your routing decision as JSON: {"agent": "<n>", "query": "<sub-query>"}
"""

NUTRITION_ANALYST_PROMPT = """You are the NutritionAnalyst sub-agent.
Receive a specific nutrition question and answer it with precision.
Cite nutrient reference values (e.g. NHS/RDA) where applicable.
Output: a structured analysis with headings: Overview, Key Nutrients, Recommendation.
"""

MEAL_PLANNER_PROMPT = """You are the MealPlanner sub-agent.
Create practical, time-efficient meal plans that match the user's goals.
Output: a markdown table with Day, Meal, Key Ingredients, Approx. Protein (g).
"""

print("Orchestrator prompt loaded.")
print(f"Sub-agents defined: NutritionAnalyst, MealPlanner")


## 3.3 · Building a Prompting Framework

The four-dimension framework — **identity, strategy, capabilities, collaboration** —
makes prompt design systematic and reusable.
Each dimension addresses a specific failure mode:

| Dimension | Failure it prevents |
|---|---|
| Identity | Ambiguous persona; generic rather than domain-specific reasoning |
| Strategy | Brittle conclusions from agents that rush to specifics |
| Capabilities | Hallucinated tool use; agents estimating instead of querying |
| Collaboration | Broken handoffs; agents acting outside their scope |


In [ ]:
class PromptTemplate:
    """
    A lightweight template with {{variable}} placeholder syntax.
    Uses string concatenation in render() — not f-strings — to avoid
    brace-escaping ambiguity when constructing the placeholder.
    """
    def __init__(self, template: str):
        self._template  = template
        self._variables = re.findall(r"\{\{(\w+)\}\}", template)

    @property
    def variables(self) -> list[str]:
        return self._variables

    def render(self, **kwargs: Any) -> str:
        result = self._template
        for key, value in kwargs.items():
            # String concatenation — not f-string — to avoid escaping errors
            placeholder = "{{" + key + "}}"
            result = result.replace(placeholder, str(value))
        missing = [v for v in self._variables if "{{" + v + "}}" in result]
        if missing:
            raise ValueError(f"Missing template variables: {missing}")
        return result

# Verify correct substitution — catches escaping bugs immediately
assert PromptTemplate("Hello {{name}}!").render(name="Alice") == "Hello Alice!"
print("PromptTemplate assertion passed.")

ASSESSMENT_TEMPLATE = PromptTemplate("""
Conduct a nutrition assessment for the following user profile:
  Name:           {{name}}
  Age:            {{age}}
  Weight (kg):    {{weight_kg}}
  Goals:          {{goals}}
  Typical Day:    {{typical_day}}
  Restrictions:   {{restrictions}}

Follow the four-step Nutrition Assessment Protocol from your Skill file.
""")
print("Template variables:", ASSESSMENT_TEMPLATE.variables)


In [ ]:
# ── 3.3.3  Fact-Checker Agent — four dimensions annotated ──────────────────
#
# Each section of the backstory maps to one prompt dimension.
# The comments are not documentation — they are the design specification
# that any engineer reading this code should be able to verify.

FACT_CHECKER_BACKSTORY = (
    # IDENTITY: who the agent is and the scope of its role
    "You are a precision nutrition scientist. Your purpose is to verify "
    "factual accuracy of dietary claims, not to generate content. "
    # STRATEGY: how it reasons and when to stop
    "Extract each factual claim. Verify or refute it using the database. "
    "Stop once all claims are checked or a high-confidence verdict is reached. "
    # CAPABILITIES: which tools and how to choose between them
    "Use lookup_nutrition for all numerical values. "
    "Never estimate nutrient values from memory. "
    # COLLABORATION: handoff format and explicit prohibitions
    "Return results to Coach Advisor as JSON. "
    "Escalate conflicting data to the Orchestrator. "
    "You are never responsible for publishing. Verify only."
)

print("Fact-Checker backstory dimensions:")
for label in ["IDENTITY", "STRATEGY", "CAPABILITIES", "COLLABORATION"]:
    print(f"  [{label}] present: True")


## 3.4 · Agent Skills: Reusable Procedural Knowledge

A Skill is a reusable procedural knowledge file stored in `SKILL.md`.
It specifies the steps an agent follows to solve a problem —
not what facts it knows or what tools it can call.

| | Tool (`lookup_nutrition`) | Skill (`SKILL.md`) |
|---|---|---|
| Nature | Declarative — what to call | Procedural — how to think |
| Executed by | Runtime | Model |
| Provides | External data access | Structured decision-making |

**Tools without Skills** → raw data with no reasoning structure.
**Skills without Tools** → structured reasoning with no verified data.
**Together** → grounded, structured decision-making.

In [ ]:
# Display the loaded Skill to inspect its four steps
skill = load_skill()
if skill:
    print(f"Skill loaded: {len(skill.splitlines())} lines")
    # Show the step headings
    steps = [l for l in skill.splitlines() if l.startswith("### Step")]
    print("\nProtocol steps:")
    for s in steps:
        print(" ", s)
else:
    print("SKILL.md not found — run from chapter_03_prompting/ directory")


## 3.5 · Prompt Optimisation with DSPy

Manual prompt engineering has a ceiling:
performance shifts with wording, model updates, and input distribution changes.
DSPy replaces the manual iteration cycle with a systematic optimisation loop:

1. **Define** the task through a Signature (inputs → outputs)
2. **Instrument** it with a metric (the NutritionJudge)
3. **Optimise** automatically (MIPROv2)
4. **Evaluate** on held-out data before deploying

> **Note:** `optimizer.compile()` makes multiple LLM calls and takes 1–5 minutes.
> Run the cells in this section when you are ready to wait.

In [ ]:
# LM Setup — two models: a lighter one for the agent, a stronger one to judge it.
# The evaluator must always be at least as capable as the system being evaluated.
main_lm = dspy.LM(
    model="openai/gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
)
main_lm("Hello", max_tokens=10)   # verify connection before proceeding
dspy.settings.configure(lm=main_lm)

judge_lm = dspy.LM(
    model="openai/gpt-4o",         # stronger model for evaluation
    api_key=os.getenv("OPENAI_API_KEY"),
)
print("LMs configured. Main: gpt-4o-mini | Judge: gpt-4o")


In [ ]:
@dspy.Tool
def foodDBtool(food_item: str) -> str:
    """Look up nutrition data from the local database.
    Returns a structured 'not found' message — never raises —
    so the agent handles missing data explicitly rather than hallucinating.
    """
    with open("foodDB.json", "r") as f:
        nutrition_db = json.load(f)
    query = food_item.lower().strip()
    if query in nutrition_db:
        info = nutrition_db[query]
        return (
            f"Nutrition for {info['serving_size']} of {query.capitalize()}: "
            f"{info['calories']} calories, {info['protein_g']}g protein, "
            f"{info['carbs_g']}g carbs, {info['fat_g']}g fat, {info['fiber_g']}g fiber."
        )
    return f"Sorry, nutrition information for '{food_item}' was not found."

# Test the tool directly
print(foodDBtool("chicken breast"))
print(foodDBtool("mystery food"))   # shows graceful not-found message


In [ ]:
class DietAnalysis(dspy.Signature):
    """Analyze a meal and provide nutritional breakdown."""
    meal: str = dspy.InputField(
        desc="Description of the meal eaten"
    )
    analysis: str = dspy.OutputField(
        desc=(
            "Nutritional breakdown with calories, protein, carbs, fat "
            "and health assessment. Acknowledge missing items explicitly."
        )
    )

# Zero prompt strings written — DSPy generates the prompt from the
# Signature and tool docstrings. This is the shift from prompting to programming.
diet_agent = dspy.ReAct(DietAnalysis, tools=[foodDBtool])
diet_agent.set_lm(main_lm)
print("diet_agent configured.")


In [ ]:
class NutritionJudge(dspy.Signature):
    """Judge the quality of a nutritional analysis."""
    meal: str          = dspy.InputField()
    analysis: str      = dspy.InputField()
    quality_score: float = dspy.OutputField(
        desc="Score 0-1 based on accuracy, completeness, and helpfulness"
    )

judge = dspy.ChainOfThought(NutritionJudge)
judge.set_lm(judge_lm)

def nutrition_metric(gold, pred, trace=None) -> float:
    result = judge(meal=gold.meal, analysis=pred.analysis)
    return result.quality_score

# Seven examples — the last one deliberately includes a missing food item.
# This teaches the agent to acknowledge gaps rather than fabricate values.
trainset = [
    dspy.Example(
        meal="Grilled chicken breast with steamed broccoli and brown rice",
        analysis="Meal Breakdown:\n- Chicken breast (100g): 165 cal, 31g protein..."
    ).with_inputs("meal"),
    dspy.Example(
        meal="Two scrambled eggs with two slices of toast",
        analysis="Meal Breakdown:\n- Eggs (2): 140 cal, 12g protein..."
    ).with_inputs("meal"),
    dspy.Example(
        meal="Apple and banana for snack",
        analysis="Meal Breakdown:\n- Apple: 95 cal, 0.5g protein..."
    ).with_inputs("meal"),
    dspy.Example(
        meal="100g salmon with 1 cup of pasta",
        analysis="Meal Breakdown:\n- Salmon (100g): 208 cal, 20g protein..."
    ).with_inputs("meal"),
    dspy.Example(
        meal="Salad with 100g chicken breast, 1 cup spinach, 1 tbsp olive oil",
        analysis="Meal Breakdown:\n- Chicken: 165 cal, 31g protein..."
    ).with_inputs("meal"),
    dspy.Example(
        meal="Bowl of oats with banana and a quarter cup of almonds",
        analysis="Meal Breakdown:\n- Oats (1 cup): 150 cal, 6g protein..."
    ).with_inputs("meal"),
    dspy.Example(
        meal="Greek yogurt with a tablespoon of honey",
        analysis="Greek yogurt (100g): 97 cal, 17g protein. Honey: not found in database."
    ).with_inputs("meal"),
]
print(f"Training set: {len(trainset)} examples")


In [ ]:
# auto='light' uses a smaller compute budget — suitable for development.
# auto='medium' or 'heavy' runs longer and typically yields higher scores.
optimizer = MIPROv2(metric=nutrition_metric, auto="light")
optimized_agent = optimizer.compile(
    diet_agent,
    trainset=trainset,
    requires_permission_to_run=False,
)
print("Optimisation complete.")


In [ ]:
result = optimized_agent(meal="pasta, 4 eggs and a banana")
print(result.analysis)


In [ ]:
# Inspect the full reasoning trace — shows tool calls and fallback strategies
optimized_agent.inspect_history()


## 3.6 · Structured Outputs

Free text breaks production. A nutrition analysis that returns
`'Calories: 200kcal'` in one run and `'~200 cal'` in another cannot be
parsed reliably by a downstream agent.

Structured outputs replace ambiguity with contracts:
- **JSON Schema** enforces types and required fields at the prompt level
- **Pydantic** enforces them at the Python level
- The **repair loop** catches and corrects malformed outputs before they propagate

> **Security note:** Tool results included in prompts can carry adversarial content
> designed to override the agent's system prompt (context poisoning).
> Truncating tool results to a maximum length (2,000 chars) limits — but does not
> eliminate — this risk. Chapter 10 covers the full threat model.

In [ ]:
STRUCTURED_ASSESSMENT_SCHEMA = {
    "type": "object",
    "properties": {
        "baseline_summary": {"type": "string"},
        "deficits": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "nutrient":  {"type": "string"},
                    "severity":  {"type": "string", "enum": ["low", "medium", "high"]},
                    "note":      {"type": "string"},
                },
                "required": ["nutrient", "severity", "note"],
            },
        },
        "priority_actions": {"type": "array", "items": {"type": "string"}},
        "goal_this_week":   {"type": "string"},
    },
    "required": ["baseline_summary", "deficits", "priority_actions", "goal_this_week"],
}

# Pydantic mirrors the schema — validation happens at the Python layer
class NutritionDeficit(BaseModel):
    nutrient:  str
    severity:  str = Field(..., pattern="^(low|medium|high)$")
    note:      str

class NutritionAssessmentResult(BaseModel):
    baseline_summary:  str
    deficits:          list[NutritionDeficit]
    priority_actions:  list[str]
    goal_this_week:    str

print("Schema and Pydantic models defined.")
print("Required fields:", STRUCTURED_ASSESSMENT_SCHEMA["required"])


In [ ]:
_repair_client = OpenAI()

def repair_output(output: dict, schema: dict, max_retries: int = 3) -> dict:
    """
    Use a separate LLM call to correct malformed JSON.
    Raises ValueError after max_retries — never loops indefinitely.
    A hard failure here is preferable to propagating bad data.
    """
    for attempt in range(max_retries):
        prompt = (
            f"Fix this JSON to match the schema exactly.\n"
            f"Schema:\n{json.dumps(schema, indent=2)}\n\n"
            f"Malformed JSON:\n{json.dumps(output, indent=2)}\n\n"
            "Return ONLY the corrected JSON. No explanation. No markdown."
        )
        resp = _repair_client.chat.completions.create(
            model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
            messages=[{"role": "user", "content": prompt}],
        )
        raw = resp.choices[0].message.content.strip()
        import re as _re
        if raw.startswith("```"):
            raw = _re.sub(r"^```(?:json)?\n?", "", raw).rstrip("`").strip()
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            continue
    raise ValueError(f"repair_output failed after {max_retries} attempts")


def run_structured_assessment(user_profile: dict) -> dict:
    """Request a schema-conforming JSON assessment, with repair on failure."""
    import re as _re
    client = OpenAI()
    prompt = (
        f"User profile: {json.dumps(user_profile, indent=2)}\n\n"
        "Conduct a nutrition assessment following your SKILL protocol. "
        "Return ONLY valid JSON matching this schema:\n"
        f"{json.dumps(STRUCTURED_ASSESSMENT_SCHEMA, indent=2)}\n"
        "No markdown fences. No extra keys. Pure JSON."
    )
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        temperature=0.2,
        messages=[
            {"role": "system", "content": build_system_prompt()},
            {"role": "user",   "content": prompt},
        ],
    )
    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = _re.sub(r"^```(?:json)?\n?", "", raw).rstrip("`").strip()
    result = json.loads(raw)
    try:
        validated = NutritionAssessmentResult.model_validate(result)
    except ValidationError:
        repaired  = repair_output(result, STRUCTURED_ASSESSMENT_SCHEMA)
        validated = NutritionAssessmentResult.model_validate(repaired)
    return validated.model_dump()

print("repair_output and run_structured_assessment defined.")


## 3.7 · The Full Pattern: Skills + Structured Outputs

This section ties everything together.

| Layer | Governs | Defined in |
|---|---|---|
| Skill | HOW the agent reasons | `SKILL.md` → injected via system prompt |
| Schema | WHAT the agent returns | `STRUCTURED_ASSESSMENT_SCHEMA` + Pydantic |
| Context | WHAT the agent sees | Chapter 5 (memory systems) |

Skills without schemas → right reasoning, unparseable output.
Schemas without Skills → right format, inconsistent reasoning.
Together they make the agent reliable end-to-end.

In [ ]:
def run_skill_guided_assessment(user_message: str, user_profile: dict) -> dict:
    """
    Section 3.7: Full pattern — Skills + Structured Outputs.

    Skill  → HOW to reason (four-step nutrition assessment protocol)
    Schema → WHAT to return (validated JSON with fixed fields)

    These two layers are independently modifiable:
    - Update the Skill (change the protocol) without touching the schema.
    - Update the schema (add a field) without touching the Skill.
    """
    import re as _re
    skill  = load_skill()
    system = build_system_prompt(skill)

    prompt = (
        f"User profile: {json.dumps(user_profile, indent=2)}\n\n"
        f"User message: {user_message}\n\n"
        "Follow the four-step Nutrition Assessment Protocol in your Skill. "
        "Return your response as valid JSON matching this schema:\n"
        f"{json.dumps(STRUCTURED_ASSESSMENT_SCHEMA, indent=2)}\n"
        "No markdown fences. Pure JSON only."
    )

    client   = OpenAI()
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        temperature=0.2,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt},
        ],
    )

    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = _re.sub(r"^```(?:json)?\n?", "", raw).rstrip("`").strip()

    result = json.loads(raw)
    try:
        validated = NutritionAssessmentResult.model_validate(result)
    except ValidationError:
        repaired  = repair_output(result, STRUCTURED_ASSESSMENT_SCHEMA)
        validated = NutritionAssessmentResult.model_validate(repaired)
    return validated.model_dump()


# Demo — requires OPENAI_API_KEY in .env
sample_profile = {
    "name": "Alex",
    "age": 34,
    "weight_kg": 78,
    "goals": "lose 5kg over 3 months",
    "typical_day": "skip breakfast, large lunch, snack at 3pm, dinner at 8pm",
    "restrictions": "no dairy",
}
assessment = run_skill_guided_assessment(
    user_message="Assess my diet and give me one thing to change this week.",
    user_profile=sample_profile,
)
print(json.dumps(assessment, indent=2))
